# Day 29 — Restart: Push Your Work, Finish Day 4 & Your First pytest File (22 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: get every piece of work on your laptop onto GitHub, finish Day 4's contact store until a real test suite passes, and write an honest status line for every day so far. Tomorrow's thin-sync project builds on exactly this: files, JSON, errors and pytest.

- Deliverable: day04/ finished (contact_store.py, main.py, dsa.py) and pushed; day29/test_day04.py green with 21 passed (18 given tests plus 3 you write); day29/status.md filled in; python dsa.py in day29/ printing all 12 checks passed.

- Where this sits: Week 5, day 1 of 7, the consolidation week decided on Day 28. Phase 1 (Foundation) runs one week longer, Phase 2 (Claude API) starts on Day 36, and applications start on 27 Dec.

Time split (5h): the Week 5 split. Short refresher, long build, and a push at the end of every block.

| Block | Time | What you do |
|---|---|---|
| Refresher | 1h | Concepts 1 to 5, typing each example in a scratch folder |
| Build | 3h | Part A push (30 min), Part B Day 4 green (2h), Part C your 3 tests and status.md (30 min) |
| DSA | 0.5h | day04/dsa.py: top words, balanced brackets; then day29/dsa.py checks |
| Push and notes | 0.5h | Final push, Remember cells in the notebook, Day 29 review in chat |

Open items first: I checked the repo at 7:45 AM IST. It still has two commits, the last from 28 Oct. Days 1 to 3 are finished, day04/ is the starter files with 8 functions raising NotImplementedError, and nothing from Days 5 to 28 is there. The Day 28 reversal check (day27/content-sync-api on main with its tests) did not pass, so the consolidation week stands. Today is designed so that one focused day changes that picture.

## Recap (2 minutes, from memory)

- In the Day 28 review file, GET /entries/search returned 404 "entry not found". Why, and what is the fix?

- Why must a webhook signature be checked over the raw request body, and not over the JSON after you parse it?

- What did the Day 28 checkpoint decide, and which single habit is Week 5 built to test?

Answers are on the last page of today's session.

## Concept 1: Getting every local folder onto GitHub

### Why it matters for an FDE

Work that is not pushed does not exist for anyone but you. A hiring manager opens your GitHub before your CV. A customer's team reviews your integration code in their repo, and a teammate takes over your branch when you are on leave. If the code sits on one laptop, none of that can happen.

FDEs often work on customer machines or through a VPN that can drop at any time. The habit that protects you is simple: commit small, push at the end of every working block, and check on the website that it arrived.

### How it works

Your work lives in three places. The working folder is the files you edit. The local repository holds the commits you made on your laptop. origin is the copy on GitHub. git status compares the first two, and after git fetch it also tells you how your commits compare with GitHub's. It is the React DevTools of git: look before you change anything.

```bash
# Step 1: see where you stand. Nothing here changes any file.
cd ~/fde-journey          # your repo folder (use your own path)
git status                # untracked folders = never committed
git log --oneline -5      # your last commits, newest first
git fetch origin          # download GitHub's view; changes nothing local
git status                # now says "ahead", "behind" or "up to date"
# "ahead of 'origin/main' by 3 commits" = committed, NOT pushed yet
git log --oneline origin/main..main   # exactly which commits those are
```

Since Day 15, each session's starter files reached you as two attachments: a patch file and a zip of the day folder. Use the zip route. I tested the patch route this morning on a copy of your repo: every patch was made against the same old main and each one edits the same README table, so after the first patch, git am stops with a merge conflict in README.md. The zip route only adds a folder, so it cannot conflict.

```bash
# Step 2, the zip route: one day folder at a time, one commit per day.
unzip -l ~/Downloads/day05.zip       # look inside first: expect day05/...
unzip -n ~/Downloads/day05.zip -d .  # -n: never overwrite a file you have
git status                           # expect: ?? day05/
git add day05                        # stage ONLY that folder
git commit -m "Day 5: classes and dataclasses (starters)"
# repeat for each day you have, oldest first, then push once
git push origin main
# GOTCHA: never "git push --force" after a rejected push. Run
# "git pull --rebase origin main" and push again instead.
```

If you did write code for some of those days, your version is already in the folder, and unzip -n leaves it alone. Commit oldest day first, so the history reads in order. Update the README table once, at the end, by hand.

```bash
# Step 3: keep junk out of git, then check what GitHub will receive.
# .gitignore lists paths git should never track (one pattern per line)
printf '.venv/\n__pycache__/\n.pytest_cache/\nday04/data/\n' >> .gitignore
git add .gitignore
git commit -m "Ignore venvs, caches and generated data"
git push origin main
git status        # -> "Your branch is up to date with 'origin/main'."
# FDE TIP: open the repo on github.com and check the latest commit date.
# If GitHub does not show it, the customer (or recruiter) cannot see it.
```

### Real-world FDE example

On a Contentstack to search-index migration, you write the migration scripts in the customer's GitHub organisation. Their platform lead reviews a pull request every afternoon and their security team scans every push. An engineer who keeps two weeks of work local and then opens one 4,000-line pull request gets it sent back unread. Small daily pushes are how the customer learns to trust you.

### Common mistakes

- git add . from the wrong folder, which stages your home folder or a whole .venv/. Run git status before every commit.

- Committing generated files such as day04/data/contacts.json or __pycache__/. They change on every run and fill diffs with noise.

- git push --force after a rejected push. It replaces GitHub's history with yours and can delete work. Pull with rebase instead.

- One giant commit called "all days". Nobody can review it, and you cannot undo one day without undoing all of them.

- unzip without -n, which silently overwrites the newer version of a file you had already edited.

## Concept 2: Validating one record at the boundary

### Why it matters for an FDE

The boundary is where outside data enters your code: a CRM export, a webhook body, a CSV a customer emails you. That data is always messy, and an FDE's job is to clean it once, right there, so every function after the boundary can trust what it gets. Day 4's validate_contact() is exactly this, and so is every Pydantic model you wrote in Week 4.

Think of the validation rules on a Contentstack content type: required fields, a regex on a URL field, a number range. Those rules protect entries created in the CMS. Data arriving from another system skips them, so you write the same rules in code.

### How it works

A good validator makes four moves. It reads safely, with .get() and or "", so a missing key or a None never crashes it. It normalises: strip spaces, lowercase emails. It checks one rule at a time and raises one specific error whose message a person can act on. And it returns a new dict, so the caller's data is never changed. The example uses CMS asset records, a different shape from your contacts, so you practise the pattern without seeing your task's answer.

In [ ]:
# Validate one CMS asset record at the boundary, then a whole export.
class InvalidAssetError(ValueError):
    """One asset record broke a rule. "is a" ValueError, so callers can
    catch either name."""
 
 
def validate_asset(raw: dict) -> dict:
    """Return a cleaned COPY of one asset, or raise InvalidAssetError."""
    # .get() returns None for a missing key instead of raising KeyError;
    # "or ''" turns None into "" so .strip() is always safe
    title = (raw.get("title") or "").strip()
    if not title:
        raise InvalidAssetError("title is empty")
    url = (raw.get("url") or "").strip()
    if not url.startswith("https://"):
        # !r puts the ORIGINAL value in quotes, so a blank shows as ''
        raise InvalidAssetError(f"url {raw.get('url')!r} is not https")
    size = raw.get("size_kb")
    # GOTCHA: True is an int in Python (True == 1), so reject bool first
    if isinstance(size, bool) or not isinstance(size, int) or size <= 0:
        msg = f"size_kb must be a positive int, got {size!r}"
        raise InvalidAssetError(msg)
    # a NEW dict: the caller's raw record is never changed
    return {"title": title, "url": url, "size_kb": size}
 
 
export = [
    {"title": " Hero banner ", "url": "https://cdn.acme.in/hero.png",
     "size_kb": 240},
    {"title": "", "url": "https://cdn.acme.in/x.png", "size_kb": 10},
    {"title": "Logo", "url": "http://cdn.acme.in/logo.svg", "size_kb": 4},
    {"title": "Promo", "url": "https://cdn.acme.in/p.mp4", "size_kb": True},
]
 
clean, errors = [], []
# enumerate(..., start=1): numbers that match what a human counts
for number, record in enumerate(export, start=1):
    try:
        clean.append(validate_asset(record))
    except InvalidAssetError as err:  # catch only the error you expect
        errors.append(f"record {number}: {err}")  # str(err) is the message
 
print(clean)
# -> [{'title': 'Hero banner', 'url': 'https://cdn.acme.in/hero.png',
#      'size_kb': 240}]
print(*errors, sep="\n")
# -> record 2: title is empty
# -> record 3: url 'http://cdn.acme.in/logo.svg' is not https
# -> record 4: size_kb must be a positive int, got True
print(repr(export[0]["title"]))  # -> ' Hero banner ' (input untouched)

Two details carry most of the value. First, the loop wraps only the call that can fail, and catches only InvalidAssetError: one bad record becomes one line in errors, and the import keeps going. Second, {size!r} prints the value the way Python would type it, so a blank shows as '' and True shows as True. That is what lets the person fixing the source data find the row.

GOTCHA: bool is a subclass of int in Python, so isinstance(True, int) is True. JSON true in a numeric field passes a plain int check. JavaScript has the opposite habit: there, typeof true is "boolean" and never "number".

### Real-world FDE example

A retailer's marketing team sends a 40,000-row contact export from their old CRM before a CMS personalisation launch. 300 rows are bad. Crashing on row 2 blocks the launch, and silently dropping 300 rows loses customers nobody will ever look for. The right output is a clean file plus a reject report such as "record 1832: email 'asha.acme.in' has no '@'", which their CRM admin can fix in an afternoon.

### Common mistakes

- Changing the input: raw["name"] = raw["name"].strip() edits the caller's dict, and a second run over the same list behaves differently.

- except Exception in the loop. A typo such as emial then becomes "record 3: name 'emial' is not defined" on every row, which looks like bad data instead of a bug.

- Error messages without the value. "invalid email" makes someone search 40,000 rows; "email 'x' has no '@'" does not.

- raw["email"].strip(), which crashes twice: KeyError when the key is missing, AttributeError when it is None.

## Concept 3: Errors that travel up: one family, one catch

### Why it matters for an FDE

Most integration code runs with nobody watching: a nightly sync started by a scheduler, a container restarted at 3 AM. When it fails, the reader is an on-call engineer at the customer, not you. They need one clear line that names the file and the problem, and an exit code that tells the scheduler the run failed. They do not need a 40-line traceback, and they must never get a job that printed an error and still exited 0.

### How it works

Errors move up the call stack until something catches them, like a thrown error in JavaScript bubbling up to the nearest catch. Good programs decide where each kind of error is caught. Three layers do it:

- Low-level code raises specific errors. When a library error means something in your domain, wrap it: raise StateFileError(msg) from err. The from keeps the original error as __cause__, so nothing is lost for debugging.

- A family base class. StateFileError is a SyncError, so code that catches SyncError catches every member, including ones you add next month. It works like instanceof with a base class in JavaScript.

- The top catches the family once. main() prints one line to stderr and returns 1. Everything below it can just raise.

In [ ]:
# Errors that travel up: one family, "raise from", one catch at the top.
import json
import sys
from pathlib import Path
 
 
class SyncError(Exception):
    """Base class: anything wrong with the sync job's own files."""
 
 
class StateFileError(SyncError):
    """The saved sync state is unreadable. Is a SyncError."""
 
 
def load_state(path: Path) -> dict:
    """Read the last sync cursor; a missing file means 'never synced'."""
    try:
        with open(path, encoding="utf-8") as f:  # closes even on error
            return json.load(f)  # like JSON.parse() on the file's text
    except FileNotFoundError:
        return {"version": 1, "cursor": None}  # first run: start fresh
    except json.JSONDecodeError as err:
        # FDE TIP: name the file AND the line, so the reader can fix it
        msg = f"{path} is not valid JSON (line {err.lineno})"
        raise StateFileError(msg) from err  # keeps the original as cause
 
 
def main(state_path: Path) -> int:
    """Run one sync. Return the exit code: 0 ok, 1 failed."""
    try:
        state = load_state(state_path)
        print("resuming from cursor", state["cursor"])
        return 0
    except SyncError as err:  # catches StateFileError too (the family)
        print(f"error: {err}", file=sys.stderr)  # errors go to stderr
        return 1
 
 
scratch = Path("scratch")
scratch.mkdir(exist_ok=True)
state_file = scratch / "state.json"
 
state_file.unlink(missing_ok=True)  # make sure the first run has no file
print(main(state_file))
# -> resuming from cursor None
# -> 0
state_file.write_text('{\n  "version": 1,\n  "cursor": "c-42"\n',
                      encoding="utf-8")  # broken: the closing } is missing
print(main(state_file))
# -> error: scratch/state.json is not valid JSON (line 4)
# -> 1
 
try:
    load_state(state_file)
except StateFileError as err:
    # __cause__ is the error you passed to "from": nothing is lost
    print(type(err.__cause__).__name__)  # -> JSONDecodeError
# REMEMBER: raise specific errors low down, catch the family once at the top

Now map this onto Day 4, because it is the design decision behind your task. Day 4 has two kinds of errors. InvalidContactError is about one record: it is caught inside the import loop and becomes a line in the reject list. ContactStoreError is about a whole file: it travels up to main(), which prints "error: ..." and exits with code 1. Two kinds, two catch places. If you keep that picture in your head, the five functions become much easier to write.

### Real-world FDE example

A customer runs your Contentstack to Algolia sync as a Kubernetes CronJob every 15 minutes. One day a release leaves the state file half written. With the pattern above, the job logs "error: /state/sync.json is not valid JSON (line 4)", exits 1, Kubernetes marks the run failed and their alerting pages someone. Without it, the job either dies with a traceback nobody reads or, worse, catches the error, prints it and exits 0, and the search index is silently stale for a week.

### Common mistakes

- A bare except:. It also catches KeyboardInterrupt and SystemExit, so Ctrl+C stops working and raise SystemExit(1) is swallowed.

- Catching in every function, printing, and returning None. The caller then crashes three lines later with a confusing TypeError far from the real cause.

- Raising a new error without from err. Python still shows the old one, but labelled "During handling of the above exception, another exception occurred", which reads like a second bug.

- Printing errors to stdout. Scripts that pipe stdout into a file or another program (Day 9) then mix data and errors.

## Concept 4: Your first pytest file for real code

### Why it matters for an FDE

Acceptance criteria written in prose get argued about. Acceptance criteria written as tests either pass or fail. When you open a pull request in a customer's repo, a green run is how a stranger's change gets trusted, and in FDE coding rounds "how would you test this?" comes up almost every time. Day 10 taught the tools. Today you use them on your own code for the first time.

### How it works

pytest finds files named test_*.py and runs every function whose name starts with test_. Inside, a plain assert is the check, like expect(x).toBe(3) in Jest; when it fails, pytest prints both sides. Most tests follow Arrange, Act, Assert: build the input, call the function once, check the result. Here is a complete first test file for the asset validator from Concept 2:

In [ ]:
# A first pytest file: plain asserts, parametrize, raises, tmp_path.
import json
 
import pytest  # the test runner, like Jest in JavaScript
 
from asset_rules import InvalidAssetError, validate_asset
 
GOOD = {"title": " Hero ", "url": "https://cdn.acme.in/h.png", "size_kb": 9}
 
 
def test_good_record_is_cleaned():
    # Arrange, Act, Assert: the three steps of almost every test
    result = validate_asset(GOOD)
    assert result == {"title": "Hero", "url": "https://cdn.acme.in/h.png",
                      "size_kb": 9}
 
 
def test_input_is_not_changed():
    before = dict(GOOD)  # a copy to compare with afterwards
    validate_asset(GOOD)
    assert GOOD == before  # proves the function returned a NEW dict
 
 
# One test body, many rows: each row runs as its own test
@pytest.mark.parametrize(
    ("change", "message"),
    [
        ({"title": "  "}, "title is empty"),
        ({"url": "http://x.in/a.png"}, "is not https"),
        ({"size_kb": True}, "got True"),
        ({"size_kb": None}, "got None"),
    ],
)
def test_bad_records_are_rejected(change, message):
    raw = {**GOOD, **change}  # like {...GOOD, ...change} in JS
    # the test passes only if this exact error type is raised inside
    # GOTCHA: match is a regex searched in str(err); escape ( ) $ . with
    # re.escape() when the message contains them
    with pytest.raises(InvalidAssetError, match=message):
        validate_asset(raw)
 
 
def test_records_from_a_file(tmp_path):
    # tmp_path: a fresh empty folder per test, deleted for you later
    export = tmp_path / "assets.json"
    export.write_text(json.dumps([GOOD, GOOD]), encoding="utf-8")
    records = json.loads(export.read_text(encoding="utf-8"))
    assert [validate_asset(r)["title"] for r in records] == ["Hero", "Hero"]
 
 
def test_this_one_fails_on_purpose():
    # a wrong expectation, to show what a failure report looks like
    assert validate_asset(GOOD)["title"] == " Hero "

Run it from the folder that holds both files. The last test fails on purpose, so you can see a failure report before you meet your own:

```bash
python -m pytest -q
# -> .......F                                    [100%]
# -> >  assert validate_asset(GOOD)["title"] == " Hero "
# -> E  AssertionError: assert 'Hero' == ' Hero '
# -> 1 failed, 7 passed in 0.02s
# 8 tests: the parametrize table alone adds 4 of them
echo $?   # -> 1   (any failure makes the exit code 1)
```

Your day29/test_day04.py uses the same tools plus two more. The table shows what each one gives you and where to look for it in the file.

| Tool | What it gives you | Where in test_day04.py |
|---|---|---|
| pytest.ini pythonpath | Imports from another folder without copying files | pythonpath = ../day04 makes import contact_store work |
| tmp_path | A fresh empty folder per test, cleaned up for you | Every load and save test |
| parametrize | One test body run over a table of rows | Bad records, blank cities |
| pytest.raises | Passes only if that exact error type is raised | Rejects, broken store, SystemExit |
| monkeypatch | Swaps a value for one test, then restores it | Points main.STORE_PATH at tmp_path |
| capsys | Captures what the code printed | Checks main's summary line |

GOTCHA: run the tests from inside day29/. pytest looks for pytest.ini starting from the folder you are in, so from the repo root it never sees pythonpath = ../day04, and every test fails with ModuleNotFoundError: No module named 'contact_store'.

### Real-world FDE example

A customer asks you to change their content import script, which has no tests and which three teams depend on. Before touching it, you write characterisation tests: tests that pin down what it does today, odd parts included. Then you make your change, and any test that turns red tells you exactly which team's behaviour you just changed. The given tests in test_day04.py play the same role for your Day 4 code.

### Common mistakes

- A test that writes to the real day04/data/ folder. It passes once, then fails or corrupts your data on the next run. Use tmp_path.

- Editing a test until it passes. The test is the contract; change the code.

- Only testing the happy path. Most real bugs live in the error branches: the missing file, the blank field, the wrong version.

- Special characters in match=. It is a regular expression, so (, ), $ and . mean something; wrap the text in re.escape() or compare str(caught.value) directly, as test_day04.py does.

- A file named tests.py or a function named check_save(). pytest only collects test_*.py files and test_* functions, and it reports "no tests ran" instead of an error.

## Concept 5: Red, green, commit, and an honest status

### Why it matters for an FDE

Large unfinished work tends to stay unfinished. "90% done" can last for weeks because nobody, including you, can see what the last 10% is. A loop of small red-to-green steps, each one pushed, makes progress visible every hour. The same skill, turned outward, is the weekly status report every FDE sends to a customer, and customers judge you on its honesty.

### How it works

The loop has five steps. Run the tests and see red. Pick one function. Write the smallest code that makes its tests pass. Run just those tests until green. Commit and push. Then pick the next function. Three pytest options make the loop fast:

```bash
# Red, green, commit for Day 4, from inside day29/
python -m pytest -q -x        # -x: stop at the first failure
# -> FAILED test_day04.py::test_validate_cleans_a_good_record - ...
# -> 1 failed in 0.04s      (red: validate_contact is still a TODO)
python -m pytest -q -k validate   # -k: run only tests whose name matches
# -> 11 failed, 10 deselected in 0.06s
# ... implement validate_contact in day04/contact_store.py ...
python -m pytest -q -k validate   # aim for: 1 failed, 10 passed
# the 1 still red is test_validate_cleans_tags, your own TODO test
python -m pytest -q --lf      # --lf: rerun only the tests that failed last
git add ../day04/contact_store.py
git commit -m "Day 4: validate_contact, 10 tests green"
git push origin main          # push after every green step, not at the end
```

The status report uses three words only: done (every TODO implemented, it runs, it is pushed), partial (say which part) and not started. Every line carries evidence: a file name, a test result, a commit. For example:

| Day | Date | Topic | Status | Evidence / next step |
|---|---|---|---|---|
| 4 | 28 Oct | Files, JSON, errors | done | 21 passed; commit a1b2c3d |
| 10 | 3 Nov | Testing with pytest | partial | test_contact_store.py on laptop, not pushed |
| 13 | 6 Nov | Async Python | not started | Catch-up after Day 33 |

Your day29/status.md has this table with all 29 rows, ready to fill in.

### Real-world FDE example

Friday, week 3 of an engagement. Your status email to the customer's head of digital reads: done, webhook receiver (merged, 47 tests); partial, search sync (2 of 5 content types); not started, SSO; risk, the SSO metadata from their IT team is a week late. She can forward it to her boss unchanged. Compare "good progress this week, most items on track", which tells her nothing and which she will remember when a date slips.

### Common mistakes

- Writing all five functions before running anything, then facing 21 failures at once.

- Waiting to commit until everything is done. A commit per green step costs ten seconds and is your undo button.

- "partial" without saying which part, or "done" for code that is not pushed.

- An optimistic status. One honest "not started" now costs less than the trust you lose when the gap is found later.

## Check your understanding (predict first, then run)

Six questions, one file. Write each prediction in the notebook before you run it. Question 3 surprises most people, including people who learned Python on an older version.

In [ ]:
# Day 29 check: predict every numbered print, then run.
import json
import re
 
# 1) "or" returns the first truthy value, so which text gets stripped?
print(1, repr((None or "x").strip()), repr(("  " or "x").strip()))
 
# 2) is True an int?
print(2, isinstance(True, int), True + True)
 
# 3) which line does json report?
try:
    json.loads('{\n  "a": 1,\n}')
except json.JSONDecodeError as err:
    print(3, err.lineno)
 
# 4) pytest.raises(match=...) uses re.search: does this pattern match?
print(4, bool(re.search("cost $5", "cost $5")),
      bool(re.search(re.escape("cost $5"), "cost $5")))
 
# 5) what is the cause, and is the new error a ValueError?
class StoreError(ValueError):
    pass
 
try:
    try:
        int("x")
    except ValueError as err:
        raise StoreError("bad number in row 2") from err
except ValueError as outer:
    print(5, type(outer).__name__, type(outer.__cause__).__name__)
 
# 6) enumerate with start=1 over a list of three
print(6, list(enumerate(["a", "b", "c"], start=1))[-1])

## Hands-on task (about 3 hours): push, Day 4 green, status

Your starter files are in day29/ in the repo: test_day04.py (the tests), pytest.ini (lets them import day04/), dsa.py (DSA checks) and status.md (the table to fill). Pull first: git pull origin main.

Part A: push everything you have (30 min).

- Run the three Concept 1 steps. Write down what git status said before you changed anything; it goes in your review.

- For each day from 5 to 28 that you have locally, or as a zip, add its folder and commit it, oldest first.

- Add the .gitignore, push, and check github.com shows today's date on the latest commit.

Part B: make Day 4 green (2 hours).

- cd day29 and run python -m pytest -q. Expect 21 failed. Read one failure report in full.

- Implement validate_contact() in day04/contact_store.py. Run with -k validate until only your own TODO test is red. Commit and push.

- Then load_contacts() and save_contacts() (-k "load or save"), then import_contacts() and merge_contacts(), then main() in day04/main.py. Commit and push after each one.

- Run python day04/main.py twice from the repo root and compare with its docstring. Then break day04/data/contacts.json (delete its last line), run again and check echo $? prints 1.

Part C: your 3 tests and status.md (30 min).

- Write the three TODO tests at the bottom of test_day04.py.

- For each one, break the rule it checks in contact_store.py, watch it go red, and undo the break.

- Fill in every row of status.md and the summary, then commit and push.

Acceptance criteria

- Inside day29/, python -m pytest -q prints 21 passed.

- The 18 given tests are unchanged: git diff on test_day04.py shows changes only in your three test bodies.

- python day04/main.py matches its docstring on the first and second run; a broken store prints one line starting with "error:" and exits with code 1, with no traceback.

- Each of your three tests fails when you break the rule it checks.

- day29/status.md has a status word and evidence on all 29 rows, and the summary is filled in.

- After a test run, git status is clean: no .venv/, __pycache__/, .pytest_cache/ or day04/data/ waiting to be added.

- At least one commit per finished function, all pushed to main.

Hints (no solutions)

- validate_contact(): the order of the checks decides which message a record gets. Follow the order of the Rules comment: name, email @, email domain, city, tags. For the domain, look only at the text after the "@"; str.partition() or split("@", 1) gives it to you.

- Tags: a set removes duplicates, and sorted() turns any collection back into a sorted list.

- load_contacts() has three outcomes: no file, broken file, good file. Write the test case for each in your head first, then the try, the two except branches and the shape check.

- import_contacts(): ask which error family main() catches. A missing import file must arrive there as a member of it.

- When a test fails, read the lines starting with E first. pytest shows what your code returned next to what the test expected.

- Stuck for more than 20 minutes on one function? Commit what you have with a "WIP:" message, push, move to the next one, and say so in the review.

## DSA practice (30 min)

No new problems this week. Finish the two in day04/dsa.py, then run python dsa.py inside day29/, which checks them against 8 fixed cases, including a tie, an oversized k and a punctuation-only text.

- Top k words. Build the counts first, then sort the pairs once. Ask what key makes Python sort by count high to low and, for equal counts, by word A to Z. A tuple key can do both.

- Balanced brackets. A list works as a stack. Write down by hand what the stack holds after each character of "([)]"; the place where your trace and the rule disagree is where the check belongs.

- Then fill in my_edge_cases() with four cases of your own, expected values worked out by hand, until it prints all 12 checks passed.

## Quiz (3 quick questions)

- Your import loop uses except Exception as err instead of except InvalidContactError as err, and validate_contact() has a typo, emial. What does the run print, and why is that worse than a crash?

- Why does test_main_first_and_second_run call monkeypatch.setattr(main, "STORE_PATH", ...)? What would happen on your laptop without it?

- A teammate's status says "Days 5 to 10: mostly done". Rewrite it as status lines, and name the evidence you would ask for.

## What you achieved today

You refreshed the Day 4 and Day 10 ideas that the rest of Week 5 depends on: validating records at the boundary, errors that travel up as one family to one catch, and pytest with tmp_path, parametrize, raises and monkeypatch. You also have a safe way to get all your local work onto GitHub and an honest status of every day so far. If Part B is green and pushed, the repo shows four finished days instead of three, and the Python score moves on Day 35.

Tomorrow (Day 30): the thin-sync fetch layer: an Entry dataclass, httpx with timeouts and retries, and paginated fetching from a mock CMS, tested with MockTransport.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 29 review": what git status showed before Part A, the link to your latest commit, the last line of your pytest run, your three tests, your status.md, your six predictions next to the real output, your recap and quiz answers, and the hours you really studied today.

## Recap answers

*Your answers:*
